In [26]:
from src.utils.data_loader import load_breast_cancer_data, load_folktables_income_data, load_loan_prediction
import numpy as np
np.random.seed(5)
from src.utils.sampling import sample
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.inspection import permutation_importance

In [3]:
df, columns, target = load_folktables_income_data()

In [5]:
df[columns] = StandardScaler().fit_transform(df[columns])

In [6]:
N, R = sample(
            "less_negative_class",
            df,
            target,
            train_fraction=0.5,
            bias_fraction=0.1,
            columns=columns,
        )

In [27]:
drop_index, feature_importance = mrs(N, R, columns)

In [28]:
feature_importance

array([1.47188806e-01, 1.31143719e-01, 3.91853662e-02, 1.14334450e-02,
       6.19704266e-03, 1.70134711e-02, 8.32267052e-03, 1.61114510e-02,
       9.84977579e-03, 1.38159515e-05, 2.31502922e-02, 5.00332728e-03,
       1.41379937e-02, 4.95516034e-02, 4.26583128e-03, 3.63885160e-03,
       1.43201268e-04, 7.66466652e-04, 4.24748380e-03, 2.36594990e-04,
       1.92905918e-04, 3.95935258e-04, 4.93443361e-04, 2.94294303e-03,
       1.03765340e-04, 1.23500621e-03, 1.83168503e-03, 8.80544573e-05,
       2.57624817e-02, 3.40024333e-03, 1.64257627e-07, 8.87325598e-03,
       2.65620807e-02, 1.09618014e-02, 1.52931343e-02, 3.18125227e-02,
       3.54395335e-02, 3.96619847e-03, 3.04949329e-03, 2.62064050e-04,
       1.39309667e-02, 2.26803663e-03, 1.61734174e-03, 1.97152364e-07,
       4.80227238e-04, 3.48081853e-03, 2.93485124e-02, 2.38092321e-04,
       2.59339691e-03, 1.87462671e-03, 2.07391231e-03, 3.03195273e-04,
       4.27443145e-02, 1.50982591e-03, 2.09007324e-03, 4.71942842e-04,
      

In [24]:
from sklearn.ensemble import RandomForestClassifier


def mrs(
    N,
    R,
    columns,
    n_drop: int = 1,
    n_splits=2,
    n_repeats=5,
    class_weights="balanced",
    random_state=None,
    *args,
    **attributes,
):
    """Performs one iteration of maximum representative sampling

    :param N: Non-representative data set
    :param R: Representative data set
    :param columns: Columns names used for training
    :param n_drop: Number of samples to drop every iteration, defaults to 1
    :param cv: Number of cross-validation iterations, defaults to 5
    :param class_weights: Type of class weights, defaults to "balanced_subsample"
    :param random_state: Random state to make results reproducible
    :return: _description_
    """
    all_predictions = np.zeros(len(N))
    feature_importance_list = []
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for train_index, test_index in kf.split(N):
        N_train, N_test = N.iloc[train_index], N.iloc[test_index]
        data = pd.concat([N_train, R])
        clf = train_pu_classifier(
            data[columns],
            data.label,
            class_weight=class_weights,
            random_state=random_state,
        )
        data = pd.concat([N_test, R])
        predictions = clf.predict_proba(N_test[columns])[:, 1]
        all_predictions[test_index] = predictions
        feature_importance = permutation_importance(
            clf,
            data[columns],
            data.label,
            n_repeats=n_repeats,
            random_state=random_state,
            n_jobs=-1,
            scoring="roc_auc",
        )
        feature_importance_list.append(feature_importance.importances_mean)

    mean_feature_importance = np.mean(feature_importance_list, axis=0)
    drop_ids = np.argpartition(all_predictions, -n_drop)[-n_drop:]
    drop_index = N.index[drop_ids]

    return drop_index, mean_feature_importance

def train_pu_classifier(X_train, y_train, class_weight="balanced", random_state=None):
    """Train the positive unlabeled classifier

    :param X_train: Training features
    :param y_train: Training target
    :param class_weight: Sample weights, defaults to "balanced"
    :return: Trained positive unlabeled classifier
    """
    clf = RandomForestClassifier(
        class_weight=class_weight,
        n_estimators=100,
        n_jobs=-1,
        random_state=random_state,
    )
    clf.fit(X_train, y_train)
    return clf